# OmniFall 3 — Training

Decoding segments on demand, batching them, and handing them to a video model.

```bash
pip install 'omnifall[transformers]'
```

Needs videos: either `OMNIFALL_ROOT`, or a component prepared as in
[02_videos.ipynb](02_videos.ipynb).

In [ ]:
import torch
from torch.utils.data import DataLoader

import omnifall

## A PyTorch dataset

`load_video_dataset` combines `load(..., video=True)` with a `Dataset` that
decodes on demand.

In [ ]:
parts = omnifall.load_video_dataset("le2i-cs", num_frames=16, target_fps=15.0)
parts

`sampling="auto"` — the default — draws a **random** window inside each segment
for the train split and a **deterministic** one everywhere else. That is what
keeps evaluation reproducible across runs and epochs.

In [ ]:
for name, part in parts.items():
    print(f"{name:11s} {len(part):5d} segments  sampling={part.sampling}")

## Batching

In [ ]:
transform = omnifall.VideoTransform(mode="val")
parts = omnifall.load_video_dataset(
    "le2i-cs", num_frames=16, transform=transform,
)

loader = DataLoader(
    parts["validation"], batch_size=4,
    collate_fn=omnifall.collate_fn, num_workers=2,
)
batch = next(iter(loader))
{k: tuple(v.shape) if torch.is_tensor(v) else type(v).__name__
 for k, v in batch.items()}

`pixel_values` is `(B, T, C, H, W)`, which is what 🤗 video models expect, so
nothing needs permuting. Pass `output_format="CTHW"` if you need the
channels-first video convention instead.

## With 🤗 transformers

`trainer_dataset` builds all three splits with the right sampling and a
transform derived from the checkpoint.

In [ ]:
parts = omnifall.trainer_dataset(
    "le2i-cs",
    model_name="MCG-NJU/videomae-small-finetuned-kinetics",
    num_frames=16,
)
model = omnifall.load_model("MCG-NJU/videomae-small-finetuned-kinetics")
model.config.num_labels, model.config.id2label[1]

The classifier head is resized to OmniFall's 16 classes and randomly
initialised — that is expected, and why `ignore_mismatched_sizes=True` is set
for you.

In [ ]:
loader = DataLoader(
    parts["validation"], batch_size=2, collate_fn=omnifall.collate_fn,
)
batch = next(iter(loader))

model.eval()
with torch.no_grad():
    out = model(pixel_values=batch["pixel_values"], labels=batch["labels"])
out.logits.shape, float(out.loss)

A full fine-tune is then the ordinary `Trainer` recipe:

```python
from transformers import Trainer, TrainingArguments

trainer = Trainer(
    model=model,
    train_dataset=parts["train"],
    eval_dataset=parts["validation"],
    data_collator=omnifall.collate_fn,
    compute_metrics=omnifall.compute_metrics,
    args=TrainingArguments(
        output_dir="out",
        remove_unused_columns=False,   # required: our columns are not model kwargs
        per_device_train_batch_size=4,
        num_train_epochs=3,
    ),
)
trainer.train()
```

`remove_unused_columns=False` matters — without it `Trainer` strips the columns
the dataset needs.

## Metrics

OmniFall is heavily imbalanced, so `compute_metrics` reports balanced accuracy
next to accuracy. A model that never predicts `fall` can still score well on
accuracy alone.

In [ ]:
import numpy as np

labels = np.array(parts["validation"].dataset["label"][:200])
lazy = np.full_like(labels, omnifall.LABEL2IDX["other"])   # always predict "other"
onehot = np.eye(16)[lazy]

omnifall.compute_metrics((onehot, labels))

## Cross-domain evaluation

The point of OmniFall is measuring the drop when a model leaves the data it was
trained on. The `to-all` configs encode exactly that: train on one source, test
on every component at once.

In [ ]:
parts = omnifall.load_video_dataset(
    "le2i-to-all-cs", num_frames=16, transform=omnifall.VideoTransform(mode="val"),
)
import collections
collections.Counter(parts["test"].dataset["dataset"]).most_common()

Scoring per source dataset then tells you where a model generalises and where it
does not:

```python
preds = ...  # your model's predictions over parts["test"]
for name in set(parts["test"].dataset["dataset"]):
    mask = [d == name for d in parts["test"].dataset["dataset"]]
    ...
```

## Multiple datasets at once

`MultiOmniFallDataset` concatenates several datasets and tags each sample with
its domain, for multi-source or domain-adaptation setups.

In [ ]:
from omnifall import MultiOmniFallDataset

a = omnifall.load_video_dataset("le2i-cs", split="train", num_frames=16)
b = omnifall.load_video_dataset("caucafall-cs", split="train", num_frames=16)
multi = MultiOmniFallDataset([a, b])
multi